<a href="https://colab.research.google.com/github/hibernator11/adaptation-GLAM-notebooks/blob/main/wacad_gemini_nls_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This notebooks shows you how to reuse the code provided in this workshop, in order to adapt it for additional datasets. In particular, we employed the material created for the workshop iPRESS conference including a collection of Jupyter Notebooks reusing Web Archive content and employing AI-based methods such as Retrieval-Augmented Generation

We reused the code provided in order to apply it to the GLAM sector, in particular, the digital collection [A Medical History of British India](https://data.nls.uk/data/digitised-collections/a-medical-history-of-british-india/) made available by the National Library of Scotland.


We use a framework to adapt the code that is described in the following picture:

<img src="https://github.com/hibernator11/adaptation-GLAM-notebooks/raw/main/framework.png">

## Installing packages

In [ ]:
!pip install transformers chromadb

## Loading python packages

First, let's import the required packages for embedding and vector storage.

In [2]:
# Import embedding and vector database libraries
from sentence_transformers import SentenceTransformer
import chromadb
import os
from chromadb.utils import embedding_functions

nls-data/chroma_db


Now we configure the vector database parameters

In [3]:
res_folder = "nls-data/"

# Configuration for vector database
db_collection_name = "nls-data-foundry"  # Name for our vector collection
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"  # Pre-trained embedding model
path_chroma = os.path.join(res_folder, "chroma_db")  # Path to store the vector database
input_content_dir = os.path.join(res_folder, "nls-sample")  # Directory with cleaned text files

print(path_chroma)

nls-data/chroma_db


### Download the dataset and extract

In [4]:
import requests
import zipfile
from io import BytesIO

# Ensure directory exists
os.makedirs(input_content_dir, exist_ok=True)

# Download the zip file
response = requests.get("https://nlsfoundry.s3.amazonaws.com/text/nls-text-indiaPapers.zip")
response.raise_for_status()  # ensure download worked

# Unzip directly into res_folder
with zipfile.ZipFile(BytesIO(response.content)) as z:
    z.extractall(res_folder)

# Rename the folder
original_unzip_path = os.path.join(res_folder, "nls-text-indiaPapers")
os.rename(original_unzip_path, input_content_dir)

Optionally, you can reduce the number of files to process from the dataset, if  it is quite large. This cell keeps a random sample of five files from the dataset, and deletes the rest.

In [5]:
import random

# Optional - set seed for reproducibility
random.seed(42)

# Get all txt files in the directory
all_files = [
    f for f in os.listdir(input_content_dir)
    if f.endswith(".txt")
]

# Randomly select 5 files to keep
files_to_keep = set(random.sample(all_files, min(5, len(all_files))))

# Delete everything else
for f in all_files:
    if f not in files_to_keep:
        os.remove(os.path.join(input_content_dir, f))

ChromaDB is a vector database designed for efficient storage and retrieval of embeddings. It allows us to perform similarity searches across our corpus.

In [ ]:
# Initialize ChromaDB client and collection
client = chromadb.PersistentClient(path=path_chroma)

embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
     model_name=embedding_model
)

collection = client.get_or_create_collection(name=db_collection_name,
                                             embedding_function=embedding_func)

# Load the sentence transformer model for creating embeddings
model = SentenceTransformer(embedding_model, token=False)

# Get all cleaned text files
files = sorted([f for f in os.listdir(input_content_dir) if f.endswith(".txt")])

# Process each file and add its contents to the vector database
for fname in files:
    # Load content file
    content_file_path = os.path.join(input_content_dir, fname)
    with open(content_file_path, encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f]

    # Create embeddings for each line of text
    # This converts text into numerical vectors that capture semantic meaning
    embeddings = model.encode(lines, show_progress_bar=True, convert_to_numpy=True)

    # Add embeddings and metadata to ChromaDB
    try:
        collection.add(
          ids=[f"{fname}_{i}" for i in range(len(lines))],
          embeddings=embeddings.tolist(),
          documents=lines,
          metadatas=[{"filename": fname} for i in range(len(lines))]
        )
    except ValueError:
        print(f"⚠️ Skipped {fname} due to ValueError")

print("✅ Indexed all lines into Chroma!")

Once we have the data, we can create a search function

In [8]:
def semantic_search(query, n_results=2):
    """Perform semantic search on the corpus using vector embeddings.

    Args:
        query: The search query as a string
        n_results: Number of results to return (default: 5)

    Returns:
        Dictionary containing search results from ChromaDB
    """
    # Load the embedding model
    model = SentenceTransformer(embedding_model)

    # Connect to the vector database
    client = chromadb.PersistentClient(path=path_chroma)
    collection = client.get_or_create_collection(name=db_collection_name)

    # Convert query to embedding vector
    query_emb = model.encode([query], convert_to_numpy=True).tolist()

    # Search for similar content in the database
    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results
    )

    # Display results with metadata
    for text, meta in zip(results["documents"][0], results["metadatas"][0]):
        print(f"{meta['filename']}")
        print(f"→ {text[:]}\n")

    return results

Now, we can use the function to search

In [ ]:
# Example semantic search query
# This demonstrates finding content related to diseases without requiring exact keyword matches
#res = semantic_search("Which hospitals treated leprosy?")
#res = semantic_search("infected villages?", 2)
#res = semantic_search("can you tell me the periods of vaccination?", 5)
#res = semantic_search("major diseases", 5)
res = semantic_search("efforts to mitigate the spread of diseases", 5)

### Using Google Gemini to build an assistant

In [10]:
from google.colab import ai

def generate_answer_gemini(context, question):
    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {question}
Answer:"""

    response = ai.generate_text(prompt, model_name="google/gemini-2.5-flash", stream=False)
    return response

In [11]:
def rag_query_gemini(query, n_results=5):
    results = semantic_search(query, n_results=n_results)
    context = "\n\n".join(results["documents"][0])
    answer = generate_answer_gemini(context, query)

    print("\n=== ANSWER ===")
    print(answer)
    print("\n=== SOURCES ===")
    for meta in results["metadatas"][0]:
        print(f"- {meta['filename']} ")

In [ ]:
rag_query_gemini("efforts to mitigate the spread of diseases?")

In [ ]:
rag_query_gemini("cities in which diseases were spread?")

Title: Web Archive as data (Jupyter notebook)
Author: Gustavo Candela
Affiliation: University of Alicante
Last updated: 2026‑04‑19

Contact: gcandela@ua.es
Repository: https://github.com/iipc/WACAD